# Instancia Premier League 2025-2026

Descarga desde Football-Data la base de esta temporada y la usa como conjunto de equipos para resolver un calendario doble Round-Robin nuevo con el modelo compartido `modelo_calendario.py`.

## Qué representa esta instancia

El CSV histórico de 2025-2026 se usa para identificar los clubes de esa temporada. El modelo comprueba que detecte 20 equipos, pero no exige que el archivo tenga los 380 partidos jugados: el calendario doble Round-Robin de 38 jornadas se construye desde cero y no reproduce las fechas oficiales.

## Flujo del notebook

1. Localiza el proyecto e importa `resolver_instancia` desde `modelo_calendario.py`.
2. Descarga la base de Football-Data y obtiene de ella los equipos.
3. Resuelve dos configuraciones: Gurobi por defecto y sin cortes ni heurísticas.
4. Resume estado, objetivo, cota, gap, tiempo y nodos, y muestra el calendario generado y una muestra de los datos históricos.

Las restricciones operativas opcionales no tienen efecto mientras sus listas estén vacías. El indicador de `viajes` solo representa partidos de visita en jornadas críticas, no distancias reales.

## Licencia para ejecutar

Antes de ejecutar la celda Python que resuelve el modelo, activa la licencia desde PowerShell o la terminal de Windows con el comando que entrega Gurobi: `grbgetkey CLAVE_ENTREGADA_POR_GUROBI` (reemplaza el texto por la clave real, sin escribirla en el notebook). Después reinicia el kernel y ejecuta las celdas. Si `grbgetkey` no se reconoce, instala Gurobi Optimizer para Windows o agrega su carpeta `bin` al `PATH`. La lectura del CSV por sí sola no requiere licencia.

In [ ]:
# Evaluar la instancia 2025-2026 con GurobiPy en dos configuraciones.
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

_directorios = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
RUTA_PROYECTO = next(
    (directorio for directorio in _directorios if (directorio / "modelo_calendario.py").is_file()),
    None,
)
if RUTA_PROYECTO is None:
    raise FileNotFoundError("No se encontró modelo_calendario.py en el proyecto ni sus carpetas padre.")
sys.path.insert(0, str(RUTA_PROYECTO))
from modelo_calendario import resolver_instancia

TEMPORADA = "2025-2026"
URL_CSV = "https://www.football-data.co.uk/mmz4281/2526/E0.csv"
PARAMETROS_MODELO = {
    "clasicos": [],
    "estadios_compartidos": [],
    "equipos_europeos": [],
    "jornadas_criticas": [],
    "max_clasicos_por_jornada": 1,
    "peso_breaks": 1.0,
    "peso_viajes": 1.0,
    "tiempo_limite": 3600,
    "numero_equipos_esperado": 20,
}

configuraciones = {
    "Normal (Gurobi por defecto)": False,
    "Sin cortes ni heurísticas": True,
}
resultados = {
    nombre: resolver_instancia(
        TEMPORADA,
        URL_CSV,
        desactivar_cortes_heuristicas=desactivar_cortes_heuristicas,
        **PARAMETROS_MODELO,
    )
    for nombre, desactivar_cortes_heuristicas in configuraciones.items()
}

reporte = pd.DataFrame(
    [
        {
            "Configuración": nombre,
            "Estado Gurobi": resultado["estado"],
            "Objetivo (ObjVal)": resultado["objetivo"],
            "Cota (ObjBoundC)": resultado["cota"],
            "Gap (MIPGap)": resultado["gap"],
            "Tiempo (s)": resultado["tiempo"],
            "Nodos (NodeCount)": resultado["nodos"],
        }
        for nombre, resultado in resultados.items()
    ]
)
display(reporte)

resultado_normal = resultados["Normal (Gurobi por defecto)"]
print(f"Partidos históricos descargados: {len(resultado_normal['partidos_historicos'])}")
print(f"Breaks: {resultado_normal['breaks']} | Viajes críticos: {resultado_normal['viajes_criticos']}")
display(resultado_normal["calendario"])
display(resultado_normal["partidos_historicos"][["Date", "HomeTeam", "AwayTeam"]].head(10))